# Load test một LLM server: thông lượng, TTFT, hàng đợi và prefix cache

Một con số "30 token/giây" không nói server chịu được bao nhiêu người dùng. Bài này viết một load generator bất đồng bộ
nhỏ (đúng loại công cụ như `vllm bench serve`, `llama-bench`, k6, Locust làm), rồi đo hai server thật trên máy của bạn:
**llama.cpp server** (qua `llama_cpp.server` của `llama-cpp-python`, CPU) và **Ollama**, cùng model Qwen2.5-0.5B Q4_K_M.
Bạn sẽ thấy thông lượng có tăng theo số request đồng thời không, TTFT tăng vì đâu, hàng đợi phình thế nào khi tải gần
công suất, và prefix cache cắt TTFT bao nhiêu lần.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell `explore` khởi động server, chạy tải, in bảng; test bỏ qua chúng.
- Cần: `uv sync --group llama` (build `llama-cpp-python`, khoảng 4 phút), Ollama với
  `qwen2.5:0.5b-instruct-q4_K_M` (`ollama pull qwen2.5:0.5b-instruct-q4_K_M`), bộ `zalo-legal`. Server llama.cpp chạy
  bằng `uv run --with "starlette-context==0.5.1" python -m llama_cpp.server ...` vì nhóm `llama` của repo chưa cài phần
  phụ thuộc server; notebook tự làm việc này, bind `127.0.0.1` ở một cổng trống và tắt server khi xong. Muốn chạy tay:
  `uv run --with "starlette-context==0.5.1" python -m llama_cpp.server --model <tệp.gguf> --host 127.0.0.1 --port 8001`.
- Nếu kernel chết giữa chừng, server có thể còn chạy nền: kiểm bằng `pgrep -af llama_cpp.server`, dọn bằng
  `pkill -f llama_cpp.server`.
- Các cell `explore` chạy khoảng 10 phút trên PC của track (CPU 4 luồng cho llama.cpp). Phần Ollama giả định Ollama chạy
  model trên GPU (GPU 4GB là đủ cho model 0.5B); chỉ CPU thì số Ollama sẽ khác hẳn.
- Chấm: `uv run pytest tracks/ai-engineer/modules/20-evaluation-production/exercises/intermediate/02-serving-load-test -q`

In [ ]:
# explore: đưa gốc repo vào sys.path, tìm thư mục dữ liệu
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
ZALO = ROOT / "data" / "raw" / "zalo-legal"

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import asyncio
import json
import math
import time
from dataclasses import dataclass, field

import numpy as np

## 1. Đọc stream SSE và đo một request

Endpoint `/v1/chat/completions` với `"stream": true` (llama.cpp server, Ollama, vLLM, OpenAI) trả **Server-Sent Events**:
mỗi sự kiện là một dòng `data: <JSON>`, xen giữa là dòng trống, kết thúc bằng `data: [DONE]`. Mỗi JSON là một
`chat.completion.chunk` có `choices[0].delta.content` là một mảnh chữ (thường một token); chunk đầu thường chỉ mang
`role`, chunk cuối có thể mang `usage`.

`parse_sse_line(line) -> dict | str | None`: bỏ khoảng trắng hai đầu; dòng không bắt đầu bằng `data:` trả `None`; phần sau
`data:` (bỏ khoảng trắng) là `[DONE]` thì trả `"DONE"`; còn lại `json.loads`, JSON hỏng thì `None`.

`stream_chat(client, url, payload, t_arrival=None, clock=time.perf_counter) -> RequestRecord` (hàm `async`): ghi
`t_send = clock()` ngay khi bắt đầu; `t_arrival` mặc định bằng `t_send`. POST `{**payload, "stream": True}` bằng
`client.stream("POST", url, json=...)` và đọc từng dòng (`aiter_lines`):

- HTTP status từ 400 trở lên: `error = "HTTP <status>"`, ghi `t_end`, trả về (`ok` là `False`).
- Mỗi sự kiện là dict có ít nhất một phần tử của `choices` mang `delta.content` không rỗng (hoặc `text` không rỗng, dạng
  completions): một **token**, ghi `clock()` vào `token_times`; token đầu cũng là `t_first`. Sự kiện có `usage` không
  rỗng: lưu vào `usage`. `"DONE"`: dừng đọc.
- Hết stream: `t_end = clock()`; `ok` là `True` khi có ít nhất một token, không thì `error = "không có token"`.
- Mọi ngoại lệ (mất kết nối, timeout): `t_end = clock()`, `error = "<TênLỗi>: <thông điệp>"`, `ok` là `False`, không ném ra.

In [ ]:
@dataclass
class RequestRecord:
    t_arrival: float
    t_send: float = math.nan
    t_first: float = math.nan
    t_end: float = math.nan
    token_times: list = field(default_factory=list)
    ok: bool = False
    error: str = ""
    usage: dict = field(default_factory=dict)


def parse_sse_line(line: str):
    line = line.strip()
    if not line.startswith("data:"):
        return None
    body = line[5:].strip()
    if body == "[DONE]":
        return "DONE"
    try:
        return json.loads(body)
    except ValueError:
        return None


def _has_text(event: dict) -> bool:
    for choice in event.get("choices") or []:
        if (choice.get("delta") or {}).get("content") or choice.get("text"):
            return True
    return False


async def stream_chat(client, url: str, payload: dict, t_arrival: float | None = None,
                      clock=time.perf_counter) -> RequestRecord:
    t_send = clock()
    rec = RequestRecord(t_arrival=t_send if t_arrival is None else t_arrival, t_send=t_send)
    try:
        async with client.stream("POST", url, json={**payload, "stream": True}) as resp:
            if resp.status_code >= 400:
                await resp.aread()
                rec.error = f"HTTP {resp.status_code}"
                rec.t_end = clock()
                return rec
            async for line in resp.aiter_lines():
                event = parse_sse_line(line)
                if event is None:
                    continue
                if event == "DONE":
                    break
                if event.get("usage"):
                    rec.usage = event["usage"]
                if _has_text(event):
                    now = clock()
                    rec.token_times.append(now)
                    if len(rec.token_times) == 1:
                        rec.t_first = now
        rec.t_end = clock()
        rec.ok = bool(rec.token_times)
        if not rec.ok:
            rec.error = "không có token"
    except Exception as exc:
        rec.t_end = clock()
        rec.error = f"{type(exc).__name__}: {exc}"
    return rec

## 2. Hai kiểu sinh tải

- **Closed loop** (vòng kín): `c` "người dùng ảo", mỗi người gửi request kế tiếp ngay khi request trước xong. Tải tự giảm
  khi server chậm, nên nó đo **công suất** ở một mức đồng thời, nhưng che mất hàng đợi mà người dùng thật gặp.
- **Open loop** (vòng hở): request đến theo lịch định trước, không chờ request trước. Đây là cách người dùng thật đến;
  khi tốc độ đến gần công suất, hàng đợi phình ra (module 21 mục 4, $$\rho/(1-\rho)$$).

`poisson_arrivals(rate, n, seed=0) -> np.ndarray`: thời điểm đến (giây, tính từ 0) của `n` request theo quá trình
Poisson tốc độ `rate`: `rng = np.random.default_rng(seed)`, khoảng cách giữa hai lần đến là
`rng.exponential(1 / rate, size=n)`, thời điểm đến là tổng cộng dồn. `ValueError` khi `rate <= 0` hoặc `n < 1`.

`run_open_loop(send, arrivals, max_concurrency=None, clock=time.perf_counter) -> list` (hàm `async`): `send(i, t_arrival)`
là coroutine trả một `RequestRecord`. Lấy `t0 = clock()`; request `i` được **lên lịch** lúc `t0 + arrivals[i]`: chờ tới
thời điểm đó, rồi gọi `send(i, t0 + arrivals[i])`. Với `max_concurrency` là số, không quá chừng ấy lời gọi `send` chạy
cùng lúc (`asyncio.Semaphore`): request đến khi đã đủ chỗ thì **chờ ở phía client**, và `t_send - t_arrival` của nó chính là
thời gian xếp hàng. Trả list kết quả theo thứ tự `i`.

`run_closed_loop(send, concurrency, n_requests, clock=time.perf_counter) -> list` (hàm `async`): `concurrency` worker
cùng lấy chỉ số kế tiếp từ một bộ đếm chung (0, 1, 2...), gọi `send(i, clock())` và chờ xong mới lấy chỉ số tiếp. Mỗi
chỉ số được gửi đúng một lần; trả list kết quả theo thứ tự `i`.

In [ ]:
def poisson_arrivals(rate: float, n: int, seed: int = 0) -> np.ndarray:
    if rate <= 0 or n < 1:
        raise ValueError("rate phải dương, n >= 1")
    rng = np.random.default_rng(seed)
    return np.cumsum(rng.exponential(1.0 / rate, size=n))


async def run_open_loop(send, arrivals, max_concurrency: int | None = None, clock=time.perf_counter) -> list:
    sem = asyncio.Semaphore(max_concurrency) if max_concurrency else None
    t0 = clock()

    async def one(i, offset):
        due = t0 + offset
        delay = due - clock()
        if delay > 0:
            await asyncio.sleep(delay)
        if sem is None:
            return await send(i, due)
        async with sem:
            return await send(i, due)

    return list(await asyncio.gather(*(one(i, float(a)) for i, a in enumerate(arrivals))))


async def run_closed_loop(send, concurrency: int, n_requests: int, clock=time.perf_counter) -> list:
    results = [None] * n_requests
    indices = iter(range(n_requests))

    async def worker():
        for i in indices:
            results[i] = await send(i, clock())

    await asyncio.gather(*(worker() for _ in range(concurrency)))
    return results

## 3. Tóm tắt một lượt chạy

`summarize(records, duration_s=None) -> dict`. Chỉ request `ok` vào các phân vị và thông lượng; lỗi được đếm riêng.
`duration_s` mặc định là `max(t_end)` (bỏ `nan`) trừ `min(t_arrival)` của mọi request. Phân vị dùng `np.percentile`
(nội suy tuyến tính); danh sách rỗng cho `nan`. Trả:

| Khoá | Nghĩa |
| --- | --- |
| `n`, `ok`, `errors` | số request, số thành công, số lỗi |
| `duration_s` | thời gian của lượt chạy |
| `rps`, `tok_s` | request thành công mỗi giây; tổng số token của request thành công mỗi giây |
| `ttft_p50`, `ttft_p95` | `t_first - t_arrival` |
| `e2e_p50`, `e2e_p95` | `t_end - t_arrival` |
| `itl_p50`, `itl_p95` | khoảng cách giữa hai token liên tiếp, **gộp** mọi request |
| `queue_p50`, `queue_p95` | `t_send - t_arrival` (thời gian chờ phía client) |
| `service_p50`, `service_p95` | `t_end - t_send` (thời gian từ lúc gửi tới khi xong) |

In [ ]:
def _pct(values, q):
    return float(np.percentile(values, q)) if len(values) else math.nan


def summarize(records, duration_s: float | None = None) -> dict:
    ok = [r for r in records if r.ok]
    if duration_s is None:
        ends = [r.t_end for r in records if not math.isnan(r.t_end)]
        duration_s = max(ends) - min(r.t_arrival for r in records)
    ttft = [r.t_first - r.t_arrival for r in ok]
    e2e = [r.t_end - r.t_arrival for r in ok]
    itl = [b - a for r in ok for a, b in zip(r.token_times, r.token_times[1:])]
    queue = [r.t_send - r.t_arrival for r in ok]
    service = [r.t_end - r.t_send for r in ok]
    out = {"n": len(records), "ok": len(ok), "errors": len(records) - len(ok), "duration_s": duration_s,
           "rps": len(ok) / duration_s, "tok_s": sum(len(r.token_times) for r in ok) / duration_s}
    for name, values in (("ttft", ttft), ("e2e", e2e), ("itl", itl), ("queue", queue), ("service", service)):
        out[f"{name}_p50"], out[f"{name}_p95"] = _pct(values, 50), _pct(values, 95)
    return out

## 4. Chạy với llama.cpp server và Ollama

Harness dưới đây (không phải bài tập) tìm tệp GGUF của Ollama (`/api/show`, dòng `FROM`), khởi động `llama_cpp.server`
trên `127.0.0.1` ở một cổng trống với 4 luồng, và tắt nó trong `finally`. Mỗi request là một câu hỏi của tập ghim Zalo
Legal (40 câu đầu của `qrels/test.jsonl`), system prompt ngắn, `max_tokens` 64, `temperature` 0.

In [ ]:
# explore: harness: câu hỏi, đường dẫn GGUF, khởi động server llama.cpp
import atexit
import socket
import subprocess
from contextlib import contextmanager

import httpx
import pandas as pd

OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODEL_TAG = "qwen2.5:0.5b-instruct-q4_K_M"
SYSTEM = "Bạn là trợ lý pháp luật. Trả lời ngắn gọn bằng tiếng Việt."

order = []
with (ZALO / "qrels" / "test.jsonl").open(encoding="utf-8") as f:
    for line in f:
        q = json.loads(line)["query-id"]
        if q not in order:
            order.append(q)
with (ZALO / "queries.jsonl").open(encoding="utf-8") as f:
    qtext = {d["_id"]: d["text"] for d in map(json.loads, f)}
QUESTIONS = [qtext[q] for q in order[:40]]


def payload(i, model="local", max_tokens=64):
    return {"model": model, "max_tokens": max_tokens, "temperature": 0,
            "messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": QUESTIONS[i % len(QUESTIONS)]}]}


def gguf_path(tag):
    info = httpx.post(OLLAMA + "/api/show", json={"model": tag}, timeout=30).json()
    return next(line.split(" ", 1)[1] for line in info["modelfile"].splitlines() if line.startswith("FROM "))


def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


@contextmanager
def llama_server(model_path, *extra):
    port = free_port()
    cmd = ["uv", "run", "--with", "starlette-context==0.5.1", "python", "-m", "llama_cpp.server",
           "--model", model_path, "--n_ctx", "4096", "--n_threads", "4", "--n_threads_batch", "4",
           "--host", "127.0.0.1", "--port", str(port), *extra]
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    atexit.register(proc.terminate)
    base = f"http://127.0.0.1:{port}"
    try:
        for _ in range(240):
            try:
                if httpx.get(base + "/v1/models", timeout=1).status_code == 200:
                    break
            except httpx.TransportError:
                time.sleep(0.5)
        else:
            raise RuntimeError("server llama.cpp không lên sau 2 phút")
        yield base + "/v1/chat/completions"
    finally:
        proc.terminate()
        proc.wait(timeout=30)


async def closed_sweep(url, model, levels, n_min=16):
    rows = []
    async with httpx.AsyncClient(timeout=900, limits=httpx.Limits(max_connections=64)) as client:
        await stream_chat(client, url, payload(0, model))
        for c in levels:
            async def send(i, t_arrival):
                return await stream_chat(client, url, payload(i, model), t_arrival)
            recs = await run_closed_loop(send, c, max(n_min, 2 * c))
            rows.append({"concurrency": c, **summarize(recs)})
    return pd.DataFrame(rows)


COLS = ["concurrency", "ok", "errors", "rps", "tok_s", "ttft_p50", "ttft_p95", "e2e_p95", "itl_p50", "queue_p95"]
GGUF = gguf_path(MODEL_TAG)
print(GGUF)

In [ ]:
# explore: llama.cpp server với cấu hình mặc định, 1 và 2 request đồng thời (khoảng 40 giây)
with llama_server(GGUF) as url:
    default_run = await closed_sweep(url, "local", [1, 2])
print(default_run[COLS].round(3).to_string(index=False))

In [ ]:
# explore: llama.cpp server với --interrupt_requests false, đồng thời 1, 2, 4, 8 (khoảng 1.5 phút)
with llama_server(GGUF, "--interrupt_requests", "false") as url:
    llama_closed = await closed_sweep(url, "local", [1, 2, 4, 8])
print(llama_closed[COLS].round(3).to_string(index=False))

In [ ]:
# explore: Ollama, cùng model, đồng thời 1, 2, 4, 8 (khoảng 40 giây trên GPU)
ollama_closed = await closed_sweep(OLLAMA + "/v1/chat/completions", MODEL_TAG, [1, 2, 4, 8])
print(ollama_closed[COLS].round(3).to_string(index=False))

In [ ]:
# explore: open loop Poisson, client giữ tối đa 1 request trong server để thấy hàng đợi (khoảng 3 phút).
# Mức tải đặt theo công suất ước từ thời gian phục vụ đo ở mức đồng thời 1: 0.4, 0.75 và 1.0 lần công suất.
async def open_sweep(url, model, rates, n=30):
    rows = []
    async with httpx.AsyncClient(timeout=900) as client:
        await stream_chat(client, url, payload(0, model))
        for rate in rates:
            async def send(i, t_arrival):
                return await stream_chat(client, url, payload(i, model), t_arrival)
            recs = await run_open_loop(send, poisson_arrivals(rate, n, seed=0), max_concurrency=1)
            rows.append({"rate": rate, **summarize(recs)})
    return pd.DataFrame(rows)


with llama_server(GGUF, "--interrupt_requests", "false") as url:
    service_s = float(llama_closed.loc[llama_closed.concurrency == 1, "service_p50"].iloc[0])
    RATES = [round(f / service_s, 3) for f in (0.4, 0.75, 1.0)]
    print("thời gian phục vụ p50:", round(service_s, 2), "s; các mức tải (request/giây):", RATES)
    llama_open = await open_sweep(url, "local", RATES)
print(llama_open[["rate", "ok", "rps", "service_p50", "service_p95", "queue_p50", "queue_p95", "e2e_p95"]]
      .round(3).to_string(index=False))

In [ ]:
# explore: khoảng tin cậy bootstrap cho p95 E2E của lượt tải nặng nhất (mục 4 lý thuyết, bài Basic 01)
heavy = llama_open.iloc[-1]
print("p95 E2E ở rate", heavy.rate, "=", round(heavy.e2e_p95, 2), "giây trên", int(heavy.ok), "request")

In [ ]:
# explore: prefix cache: system prompt dài (một điều luật khoảng 1 700 token), câu hỏi khác nhau ở cuối (khoảng 2 phút)
docs = []
with (ZALO / "corpus.jsonl").open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if 5000 <= len(d["text"]) <= 6000:
            docs.append(d)
        if len(docs) == 2:
            break
ASK = ["Văn bản này quy định về vấn đề gì?", "Ai chịu trách nhiệm thi hành?", "Có mức phạt nào không?",
       "Tóm tắt khoản 1.", "Văn bản có hiệu lực khi nào?", "Đối tượng áp dụng là ai?", "Khoản cuối nói gì?"]
SEQ = [("A", 0, ""), ("A", 1, ""), ("A", 2, ""), ("B", 0, ""), ("A", 3, ""), ("B", 1, ""), ("A", 4, ""),
       ("A", 5, " "), ("A", 6, "")]


def prefix_run(url):
    rows = []
    with httpx.Client(timeout=600) as client:
        for key, qi, lead in SEQ:
            doc = docs[0] if key == "A" else docs[1]
            system = lead + "Bạn là trợ lý pháp luật. Chỉ trả lời dựa trên văn bản sau.\n\n" + doc["title"] + "\n" + doc["text"]
            body = {"model": "local", "max_tokens": 1, "temperature": 0,
                    "messages": [{"role": "system", "content": system}, {"role": "user", "content": ASK[qi]}]}
            t0 = time.perf_counter()
            usage = client.post(url, json=body).json()["usage"]
            rows.append(dict(prefix=key + (" + dấu cách đầu" if lead else ""), prompt_tokens=usage["prompt_tokens"],
                             seconds=round(time.perf_counter() - t0, 2)))
    return pd.DataFrame(rows)


with llama_server(GGUF, "--interrupt_requests", "false") as url:
    no_cache = prefix_run(url)
with llama_server(GGUF, "--interrupt_requests", "false", "--cache", "true", "--cache_type", "ram") as url:
    ram_cache = prefix_run(url)
print(no_cache.assign(ram_cache_s=ram_cache.seconds).to_string(index=False))

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Cấu hình mặc định của `llama_cpp.server` với 2 request đồng thời: bao nhiêu request thành công, các request lỗi trông
  thế nào (`error`, số token)? Vì sao đây là lỗi nguy hiểm khi load test (hãy đọc `--help` của server)?
- Khi tăng số request đồng thời, thông lượng (`rps`, `tok_s`) của hai server đổi thế nào? TTFT tăng vì sao, trong khi
  inter-token latency gần như đứng yên? Load generator của bạn có thấy hàng đợi đó ở cột `queue` không?
- Open loop: thời gian phục vụ có đổi theo tốc độ đến không? Hàng đợi p95 đổi thế nào khi tải tiến gần công suất (ước
  công suất từ `service_p50`)?
- p95 E2E của lượt nặng nhất tính trên bao nhiêu request? Dùng hàm bootstrap của bài Basic 01 để đặt khoảng tin cậy cho
  nó: khoảng rộng bao nhiêu?
- Prefix cache: lần gọi đầu và lần gọi cùng prefix chênh nhau bao nhiêu lần? Xen kẽ hai tài liệu A, B thì sao, có và
  không có `--cache`? Một dấu cách ở đầu system prompt làm gì?

Sau đó làm phần trắc nghiệm của bài trong app.